## Funciones de apoyo
Todas las consultas usan `buscar()`, que imprime siempre las mismas métricas (resultados, documentos examinados y tiempo) para poder comparar.

In [ ]:
RESULTADOS = {}      # guarda los resultados de las pruebas


def registrar(prueba, ok, detalle=""):
    RESULTADOS[prueba] = (ok, detalle)
    estado = "Correcto" if ok else "Revisar"
    if detalle:
        print(f"{estado}: {prueba} - {detalle}")
    else:
        print(f"{estado}: {prueba}")
    

def buscar(selector, use_index=None, limit=100000, titulo=""):
    # ejecuta una consulta Mango (_find) e imprime sus métricas de rendimiento.
    payload = {"selector": selector, "limit": limit, "execution_stats": True}
    if use_index:
        payload["use_index"] = [use_index, use_index]
        payload["allow_fallback"] = False
    res = SESSION.post(f"{DB_URL}/_find", json=payload, timeout=300)
    res.raise_for_status()
    data = res.json()
    stats = data.get("execution_stats", {})
    print(f"{titulo}: {len(data.get('docs', []))} resultados | "
          f"docs examinados: {stats.get('total_docs_examined')} | "
          f"{stats.get('execution_time_ms')} ms")
    if data.get("warning"):
        print("   aviso de CouchDB:", data["warning"].split(".")[0])
    return data


def guardar_design_doc(ddoc):
    # crea o actualiza un design document (vistas, update handlers, validaciones).
    url = f"{DB_URL}/{ddoc['_id']}"
    actual = SESSION.get(url, timeout=10)
    if actual.status_code == 200:
        ddoc["_rev"] = actual.json()["_rev"]      # necesario para actualizar
    SESSION.put(url, json=ddoc, timeout=60).raise_for_status()


def verificar_datos():
    # comprueba que existan productos, clientes y pedidos cargados. 
    for tipo in ("product", "customer", "order"):
        r = SESSION.post(f"{DB_URL}/_find", timeout=120,
                         json={"selector": {"type": tipo}, "limit": 1, "fields": ["_id"]}).json()
        if not r.get("docs"):
            raise RuntimeError(f"No hay documentos '{tipo}'. Ejecute primero Simulacion.ipynb.")
    print("La base de datos contiene productos, clientes y pedidos.")


verificar_datos()